# Fed-KAN week 1: metrics, parameter reduction and 8-bit heads

FedAvg over 4 LargeST districts, 15-minute data, 5 rounds. Every run is scored on the held-out test split with MAE, RMSE, MAPE and R2, overall and per forecast step, in fp32 and with the head rounded to int8.

Runtime > Change runtime type > T4 GPU. Expects `MyDrive/fedkan/fedkan_code_v3.zip` and `MyDrive/fedkan/data_real/district_*`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# runs, most important first: the core three heads, then the order and width ablations
ROUNDS = 5
RESAMPLE = 3
BATCH = 64
RUNS = [
    ('fc_h64',          '--head fc --kan-hidden 64'),
    ('taylor_o2_h64',   '--head taylor --taylor-order 2 --kan-hidden 64'),
    ('bspline_h64',     '--head bspline --grid-size 8 --spline-order 3 --kan-hidden 64'),
    ('taylor_o1_h64',   '--head taylor --taylor-order 1 --kan-hidden 64'),
    ('taylor_o3_h64',   '--head taylor --taylor-order 3 --kan-hidden 64'),
    ('taylor_o2_h32',   '--head taylor --taylor-order 2 --kan-hidden 32'),
    ('taylor_o2_h16',   '--head taylor --taylor-order 2 --kan-hidden 16'),
]
DRIVE = '/content/drive/MyDrive/fedkan'
WORK = '/content/fedkan'
OUT = f'{DRIVE}/results_week1'

In [ ]:
# copy code and data off Drive onto the local disk
import os, shutil, zipfile
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
with zipfile.ZipFile(f'{DRIVE}/fedkan_code_v3.zip') as z:
    z.extractall(WORK)
shutil.copytree(f'{DRIVE}/data_real', f'{WORK}/phase2/data_real')
os.makedirs(OUT, exist_ok=True)
print(sorted(os.listdir(f'{WORK}/phase2/data_real')))

In [ ]:
%cd /content/fedkan/phase2
COMMON = f'--data data_real --rounds {ROUNDS} --resample {RESAMPLE} --batch-size {BATCH} --eval-split test'

In [ ]:
# smoke test of the new metrics path on each head type (a few batches each)
for head in ['fc', 'taylor', 'bspline']:
    !python run_federated_local.py --head {head} --data data_real --rounds 1 --resample {RESAMPLE} --batch-size {BATCH} --max-batches 3 --max-val-batches 3 --eval-split test

In [ ]:
# full runs; each result is copied to Drive as soon as it finishes, and finished runs are skipped on a rerun
for name, flags in RUNS:
    if os.path.exists(f'{OUT}/{name}.json'):
        print('skip', name)
        continue
    !python run_federated_local.py {flags} {COMMON} --out results/{name}.json
    if os.path.exists(f'results/{name}.json'):
        shutil.copy(f'results/{name}.json', f'{OUT}/{name}.json')

In [ ]:
# summary table and per-step table, saved to Drive
import json
rows = []
for name, _ in RUNS:
    path = f'{OUT}/{name}.json'
    if not os.path.exists(path):
        continue
    with open(path) as f:
        r = json.load(f)
    fp, q = r['final']['fp32'], r['final']['int8_head']
    rows.append((name, r['head_params'], r['comm']['head_bytes_fp32'] / 1024, r['comm']['head_bytes_int8'] / 1024, fp, q, r['rounds'][-1]['minutes']))

lines = [f"{'run':16}{'head params':>12}{'head KB':>9}{'int8 KB':>9}{'MAE':>8}{'RMSE':>8}{'MAPE%':>8}{'R2':>7}{'MAE int8':>10}{'min':>7}"]
for name, n, kb, kb8, fp, q, mins in rows:
    lines.append(f"{name:16}{n:>12,}{kb:>9.1f}{kb8:>9.1f}{fp['mae']:>8.2f}{fp['rmse']:>8.2f}{fp['mape']:>8.2f}{fp['r2']:>7.3f}{q['mae']:>10.2f}{mins:>7.1f}")
lines.append('')
lines.append('MAE by forecast step (test split)')
lines.append(f"{'run':16}" + ''.join(f'{15 * (h + 1):>8} min' for h in range(5)))
for name, n, kb, kb8, fp, q, mins in rows:
    lines.append(f'{name:16}' + ''.join(f"{s['mae']:>12.2f}" for s in fp['steps']))
table = '\n'.join(lines)
print(table)
with open(f'{OUT}/summary.txt', 'w') as f:
    f.write(table + '\n')